# Document Question Answering System using Retrieval-Augmented Generation (RAG)

## Internship Assignment

### Objective
The objective of this project is to build a Retrieval-Augmented Generation (RAG) based Document Question Answering System capable of answering user questions from custom documents such as PDFs and text files.

Unlike traditional Large Language Models (LLMs), a RAG system first retrieves the most relevant information from a document and then generates an answer based only on the retrieved context. This significantly reduces hallucinations and improves factual accuracy.

---

## Workflow

The complete pipeline consists of the following stages:

1. Document Ingestion
2. Text Preprocessing
3. Text Chunking
4. Embedding Generation
5. Vector Database Creation
6. User Query Processing
7. Context Retrieval
8. Answer Generation
9. Validation
10. Performance Evaluation

---

## Expected Output

The system should:

- Load custom PDF documents
- Convert them into searchable vector embeddings
- Retrieve the most relevant document chunks
- Generate accurate answers using retrieved context
- Produce validation logs
- Produce system metrics

# Install Required Libraries

The following libraries are required for building the complete RAG pipeline.

- **PyPDF2** → Reading PDF documents
- **LangChain** → Text chunking utilities
- **Sentence Transformers** → Creating semantic embeddings
- **FAISS** → Vector similarity search
- **Transformers** → Language model for answer generation
- **Torch** → Backend for transformer models

In [ ]:
!pip -q install langchain
!pip -q install langchain-community
!pip -q install sentence-transformers
!pip -q install faiss-cpu
!pip -q install transformers
!pip -q install accelerate
!pip -q install PyPDF2
!pip -q install pypdf
!pip -q install torch

We import all the libraries..

Each library has a specific role:

- **PyPDF2** → Extract text from PDF files
- **RecursiveCharacterTextSplitter** → Divide large text into manageable chunks
- **SentenceTransformer** → Convert text into dense vector embeddings
- **FAISS** → Store and search embeddings efficiently
- **transformers** → Load the language model
- **time** → Measure execution time
- **os** → File operations

In [ ]:
import os
import time
import numpy as np
import faiss
import PyPDF2

from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer

from transformers import pipeline

from IPython.display import display, Markdown

#Load the PDF Document

The uploaded PDF document is converted into raw text so that it can be processed further.

Input:
- PDF Document

Output:
- Raw Text

In [ ]:
pdf_path = "/content/sample.pdf"

text = ""

with open(pdf_path, "rb") as pdf_file:

    reader = PyPDF2.PdfReader(pdf_file)

    total_pages = len(reader.pages)

    print("="*60)
    print("Reading PDF...")
    print("="*60)

    print(f"Total Pages : {total_pages}")

    for page in reader.pages:
        text += page.extract_text()

print("\nPDF Loaded Successfully.")

Reading PDF...
Total Pages : 1

PDF Loaded Successfully.


In [ ]:
#Extracted text stats
print("Characters Extracted :", len(text))

print("="*60)

print("\nFirst 1000 Characters\n")

print(text[:1000])

Characters Extracted : 3045

First 1000 Characters

Tanu Jain
+91 8529355289 — tanujainn2006@gmail.com — linkedin.com/in/tanu-jain-a40763296
SUMMARY
B.Tech (CSE-AI) student with strong foundation in Data structures and algorithms, Object Oriented Progrmming,
software development, web technologies, and machine learning. Experienced in building full-stack applications, com-
puter vision models, and database-driven systems. Seeking opportunities in software engineering and AI/ML
EDUCATION
B.Tech in CSE (AI), SKIT Jaipur 2023 – 2027
CGPA: 9.39
Class 12 (RBSE) – 92% 2023
Class 10 (RBSE) – 100% 2021
SKILLS
Languages:C, C++, Java, Python, SQL
Web Development:HTML, CSS, Bootstrap, JavaScript
AI/ML:Machine Learning, Computer Vision, CNN
Libraries/Frameworks:NumPy, Pandas, OpenCV, TensorFlow, Keras, Scikit-learn
Tools:GitHub, MySQL, Flask, Jupyter Notebook, VS Code
Core Concepts:Data Structures, OOP, DBMS, Operating Systems, Computer Networks
Soft Skills:Communication, Teamwork, Adaptability
EXP

#Text Cleaning

The extracted text may contain unnecessary spaces, tabs, and multiple newline characters.

Cleaning the text improves chunk quality and retrieval accuracy.

In [ ]:
clean_text = text.replace("\n", " ")
clean_text = clean_text.replace("\t", " ")
clean_text = " ".join(clean_text.split())
print("Before Cleaning :", len(text))
print("After Cleaning  :", len(clean_text))

Before Cleaning : 3045
After Cleaning  : 3045


#Text Chunking

Large Language Models (LLMs) cannot efficiently process very large documents at once. Therefore, instead of storing the complete document, we divide it into smaller overlapping chunks.

Why overlap?

Suppose one paragraph ends at the boundary of a chunk and the next paragraph starts in the next chunk. Without overlap, important contextual information may be lost.

To solve this, we use **RecursiveCharacterTextSplitter**.

Configuration used:

- Chunk Size = 500 characters
- Chunk Overlap = 100 characters

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len
)
chunks = text_splitter.split_text(clean_text)
print("Text Chunking Completed")
print("Total Chunks Created :", len(chunks))

Text Chunking Completed
Total Chunks Created : 8


In [ ]:
#Display Sample Chunks
print("First Chunk")
print(chunks[0])

print("\n")

print("Second Chunk")
print(chunks[1])

First Chunk
Tanu Jain +91 8529355289 — tanujainn2006@gmail.com — linkedin.com/in/tanu-jain-a40763296 SUMMARY B.Tech (CSE-AI) student with strong foundation in Data structures and algorithms, Object Oriented Progrmming, software development, web technologies, and machine learning. Experienced in building full-stack applications, com- puter vision models, and database-driven systems. Seeking opportunities in software engineering and AI/ML EDUCATION B.Tech in CSE (AI), SKIT Jaipur 2023 – 2027 CGPA: 9.39 Class


Second Chunk
engineering and AI/ML EDUCATION B.Tech in CSE (AI), SKIT Jaipur 2023 – 2027 CGPA: 9.39 Class 12 (RBSE) – 92% 2023 Class 10 (RBSE) – 100% 2021 SKILLS Languages:C, C++, Java, Python, SQL Web Development:HTML, CSS, Bootstrap, JavaScript AI/ML:Machine Learning, Computer Vision, CNN Libraries/Frameworks:NumPy, Pandas, OpenCV, TensorFlow, Keras, Scikit-learn Tools:GitHub, MySQL, Flask, Jupyter Notebook, VS Code Core Concepts:Data Structures, OOP, DBMS, Operating Systems, Com

#Generate Text Embeddings

Computers cannot understand raw text directly, so every text chunk is converted into a numerical vector called **Embedding**.An embedding captures the semantic meaning of a sentence.

We use:

**SentenceTransformer**

Model:
`all-MiniLM-L6-v2`

In [ ]:
print("Loading Embedding Model...")

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding Model Loaded Successfully.")

Loading Embedding Model...
Embedding Model Loaded Successfully.


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
# Generate Embeddings
print("Generating Embeddings...")

start_time = time.time()

embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True
)

end_time = time.time()

print("\nEmbedding Generation Completed.")

print(f"Time Taken : {end_time-start_time:.2f} seconds")

Generating Embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding Generation Completed.
Time Taken : 0.77 seconds


In [ ]:
# Embedding Statistics

embeddings = np.array(embeddings)

print("Embedding Shape :", embeddings.shape)

print("Number of Chunks :", embeddings.shape[0])

print("Embedding Dimension :", embeddings.shape[1])

Embedding Shape : (8, 384)
Number of Chunks : 8
Embedding Dimension : 384


#Build FAISS Vector Database

The generated embeddings are stored inside a **Vector Database**.A Vector Database enables efficient similarity search.Instead of searching words, it searches vectors that have similar semantic meaning.

We use **FAISS (Facebook AI Similarity Search)** because:

- Extremely fast
- Lightweight
- Open Source
- Optimized for dense vector search

The embedding dimension must match the output dimension of the embedding model.

For `all-MiniLM-L6-v2`, the embedding dimension is **384**.

In [ ]:
#Create FAISS Index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("Total Stored Vectors :", index.ntotal)

Total Stored Vectors : 8


#Accept User Query
Once the document has been converted into embeddings and stored in the vector database, the system is ready to answer user questions.

The user's question is also converted into an embedding using the same embedding model.

Then FAISS can efficiently retrieve the most semantically similar chunks.

In [ ]:
query = input("Enter your question : ")

print("Question :", query)

Enter your question : What is the name of the person mentioned in this resume?
Question : What is the name of the person mentioned in this resume?


# Convert Query into Embedding

To compare the user's question with the document chunks, the question must also be converted into a numerical embedding.The same embedding model (`all-MiniLM-L6-v2`) is used to ensure that both document chunks and the query exist in the same semantic vector space.

In [ ]:
query_embedding = embedding_model.encode([query])
query_embedding = np.array(query_embedding)
print("Query Embedding Shape :", query_embedding.shape)

Query Embedding Shape : (1, 384)


#Retrieve Relevant Chunks

The FAISS vector database compares the query embedding with all stored document embeddings.
The chunks with the smallest distance (highest similarity) are retrieved.

Here, we retrieve the **Top 3** most relevant chunks.

In [ ]:
# Retrieve Top-K Chunks
TOP_K = 3

distances, indices = index.search(
    query_embedding,
    TOP_K
)

#Create Context for the Language Model

The retrieved chunks are combined into a single context.

This context is then supplied to the language model along with the user's question.

Instead of answering from its own knowledge, the model answers using only the retrieved document information.

In [ ]:
retrieved_chunks = []

for idx in indices[0]:
    retrieved_chunks.append(chunks[idx])

context = "\n\n".join(retrieved_chunks)

print("Retrieved Context:")
print(context[:1000])

Retrieved Context:
Tanu Jain +91 8529355289 — tanujainn2006@gmail.com — linkedin.com/in/tanu-jain-a40763296 SUMMARY B.Tech (CSE-AI) student with strong foundation in Data structures and algorithms, Object Oriented Progrmming, software development, web technologies, and machine learning. Experienced in building full-stack applications, com- puter vision models, and database-driven systems. Seeking opportunities in software engineering and AI/ML EDUCATION B.Tech in CSE (AI), SKIT Jaipur 2023 – 2027 CGPA: 9.39 Class

•NPTEL: Psychology of Language, Soft Skills Development, Enhancing soft skills and Personality •NLP Workshop – DataPlay EXTRA-CURRICULAR ACTIVITIES •Volunteered at AI Buildathon providing technical support •Coordinated cultural events at ICI Fest •Led design team for event promotions 1

engineering and AI/ML EDUCATION B.Tech in CSE (AI), SKIT Jaipur 2023 – 2027 CGPA: 9.39 Class 12 (RBSE) – 92% 2023 Class 10 (RBSE) – 100% 2021 SKILLS Languages:C, C++, Java, Python, SQL Web Dev

# Load Question Answering Model

After retrieving the relevant document chunks, a Question Answering model is loaded.

In this project, the `deepset/roberta-base-squad2` model is used. It is trained to find answers directly from the given context.

The model receives two inputs:

1. The user's question
2. The retrieved document context

It then identifies the most relevant span of text and returns it as the final answer.

In [ ]:
!pip uninstall -y transformers
!pip install -q transformers==4.44.2 accelerate sentencepiece

Found existing installation: transformers 4.44.2
Uninstalling transformers-4.44.2:
  Successfully uninstalled transformers-4.44.2


In [ ]:
import transformers
import huggingface_hub

print("Transformers:", transformers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

Transformers: 4.44.2
Hugging Face Hub: 0.36.2


In [ ]:
from transformers import pipeline

print("Loading QA Model...")

qa_pipeline = pipeline(
    task="question-answering",
    model="deepset/roberta-base-squad2"
)

print("Model Loaded Successfully!")

Loading QA Model...


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Model Loaded Successfully!


# Generate Final Answer

The retrieved context and the user's question are passed to the Question Answering model.

Unlike traditional chatbots, the model does not answer only from its pre-trained knowledge. Instead, it uses the information retrieved from the uploaded document.

This makes the generated response more accurate and grounded in the document content.

In [ ]:
result = qa_pipeline(
    question=query,
    context=context
)
print("Answer:", result["answer"])
print("Confidence:", round(result["score"], 4))

Answer: Tanu Jain
Confidence: 0.1689


# Validation Logs

To verify that the system is working correctly, the retrieved chunks and generated answers are displayed.

These logs help in checking:

- Whether the correct document chunks were retrieved
- Whether the retrieved information is relevant to the question
- Whether the final answer matches the document content

Validation logs are useful for debugging and evaluating the overall performance of the RAG pipeline.

In [ ]:
print("=" * 70)
print("VALIDATION LOGS")
print("=" * 70)

print(f"Question: {query}")
print("\nRetrieved Chunks:")

for i, idx in enumerate(indices[0]):
    print(f"\nChunk {i+1}:")
    print(chunks[idx][:300])

print("\nFinal Answer:")
print(result)

print("=" * 70)

VALIDATION LOGS
Question: What is the name of the person mentioned in this resume?

Retrieved Chunks:

Chunk 1:
Tanu Jain +91 8529355289 — tanujainn2006@gmail.com — linkedin.com/in/tanu-jain-a40763296 SUMMARY B.Tech (CSE-AI) student with strong foundation in Data structures and algorithms, Object Oriented Progrmming, software development, web technologies, and machine learning. Experienced in building full-st

Chunk 2:
•NPTEL: Psychology of Language, Soft Skills Development, Enhancing soft skills and Personality •NLP Workshop – DataPlay EXTRA-CURRICULAR ACTIVITIES •Volunteered at AI Buildathon providing technical support •Coordinated cultural events at ICI Fest •Led design team for event promotions 1

Chunk 3:
engineering and AI/ML EDUCATION B.Tech in CSE (AI), SKIT Jaipur 2023 – 2027 CGPA: 9.39 Class 12 (RBSE) – 92% 2023 Class 10 (RBSE) – 100% 2021 SKILLS Languages:C, C++, Java, Python, SQL Web Development:HTML, CSS, Bootstrap, JavaScript AI/ML:Machine Learning, Computer Vision, CNN L

# System Metrics Report

The system metrics report provides important information about the implemented RAG pipeline.

The report includes:

- Number of generated chunks
- Chunk size and overlap
- Embedding model used
- Embedding dimensions
- Vector database information
- Number of stored vectors
- Question Answering model used

These metrics help in understanding the configuration and performance of the system.

In [ ]:
print("SYSTEM METRICS REPORT")
print("=" * 70)

print(f"Total Document Chunks      : {len(chunks)}")
print(f"Chunk Size                : 500")
print(f"Chunk Overlap             : 100")
print(f"Embedding Model           : all-MiniLM-L6-v2")
print(f"Embedding Dimension       : {embeddings.shape[1]}")
print(f"Vector Database           : FAISS")
print(f"Stored Vectors            : {index.ntotal}")
print(f"Question Answering Model  : deepset/roberta-base-squad2")

SYSTEM METRICS REPORT
Total Document Chunks      : 8
Chunk Size                : 500
Chunk Overlap             : 100
Embedding Model           : all-MiniLM-L6-v2
Embedding Dimension       : 384
Vector Database           : FAISS
Stored Vectors            : 8
Question Answering Model  : deepset/roberta-base-squad2


# Conclusion

In this project, a Document Question Answering System was developed using Retrieval-Augmented Generation (RAG).

The system extracts text from custom documents, splits it into smaller chunks, generates embeddings, and stores them in a FAISS vector database. When a user asks a question, the system retrieves the most relevant chunks and generates an answer using the retrieved context.

The system was tested on custom documents such as resumes and was able to answer questions related to names, education, CGPA, and skills.